# 01 — Entrenamiento Random Forest
**Target:** `precio_total_usd`  
**Features:** superficie, habitaciones, baños, antigüedad, zona, anillo (coincide con random_forest_service.py)


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

CSV_PATH   = '../ml_models/datasets/ML_Supervisado_Prediccion_Precio (1).csv'
MODEL_PATH = '../ml_models/trained/random_forest_v1.pkl'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)


In [ ]:
# Features alineadas con random_forest_service.py:_build_features
# [superficie_total_m2, habitaciones, banos, anio_construido, zona_enc, anillo_vial]
df['anio_construido'] = 2026 - df['antiguedad_anos']
le_zona = LabelEncoder()
df['zona_enc'] = le_zona.fit_transform(df['zona'])

FEATURES = [
    'superficie_total_m2','habitaciones','banos',
    'anio_construido','zona_enc','anillo_vial',
]
TARGET = 'precio_total_usd'

X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')


In [ ]:
# Entrenamiento
params = dict(n_estimators=200, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=42)
model = RandomForestRegressor(**params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MAE: ${mae:,.0f} | R²: {r2:.4f}')


In [ ]:
# Guardar modelo + encoder
Path('../ml_models/trained').mkdir(parents=True, exist_ok=True)
bundle = {"model": model, "le_zona": le_zona, "features": FEATURES}
joblib.dump(bundle, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')


In [ ]:
# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(fi.head(6))
